In [ ]:
import os
from typing import List
import requests
from dotenv import load_dotenv
from utils.agents import Agent
from utils.messages import BaseMessage
from utils.tooling import tool
import random
load_dotenv()

In [ ]:
@tool
def get_weather(city: str) -> dict:
  API_KEY = os.getenv("OPENWEATHER_API_KEY")
  BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

  params = {
  "q": city,
  "appid": API_KEY,
  "units": "metric"
  }

  response = requests.get(BASE_URL, params=params)
  response.raise_for_status()
  return response.json()

In [ ]:
@tool
def get_exchange_rate(from_currency: str = "USD") -> dict:
  API_KEY = os.getenv("EXCHANGERATE_API_KEY")
  BASE_URL = "https://v6.exchangerate-api.com/v6"

  url = f"{BASE_URL}/{API_KEY}/latest/{from_currency}"
  response = requests.get(url)
  response.raise_for_status()
  return response.json()

In [ ]:
@tool
def get_random_pokemon() -> dict:
  """Get a random Pokemon from the original 151"""
  URL = "https://pokeapi.co/api/v2/pokemon?limit=151"
  response = requests.get(URL)
  response.raise_for_status()
  return random.choice(response.json()['results'])

In [ ]:
tools = [get_random_pokemon,get_exchange_rate,get_weather]

In [ ]:
agent = Agent(
  model_name="gpt-4o-mini",
  instructions=(
  "You are an assistant that can help with:\n"
  "1. Getting weather information for cities\n"
  "2. Checking currency exchange rates\n"
  "3. Getting a random Pokemon\n"
  "Use the available tools to help answer questions about these topics.\n"
  "Maintain context across conversations within the same session."
  ),
  tools=tools
)

In [ ]:
session_id = "external_tools"

run1 = agent.invoke(
query="What's the weather like in London?",
session_id=session_id,
)

run2 = agent.invoke(
query="What's the exchange rate from USD to EUR?",
session_id=session_id,
)

run3 = agent.invoke(
query="Pick one random Pokemon!",
session_id=session_id,
)

In [ ]:
runs = agent.get_session_runs(session_id)
for i, run_object in enumerate(runs, 1):
print(f"\n# Run {i}", run_object.metadata)
print("Messages:")
print_messages(run_object.get_final_state()["messages"])